In [ ]:
!nvidia-smi

Wed Oct 14 20:52:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.7/79.7 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.0/30.0 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 90.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 122.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 123.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 126.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 26.4 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing 

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import unsloth
print(unsloth.__version__)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
2026.10.5


In [ ]:
# Cell 3: Load Gemma 3 1B in 4-bit
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-3-1b-it-unsloth-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

print("Model loaded successfully!")

==((====))==  Unsloth 2026.10.5: Fast Gemma3 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Model loaded successfully!


In [ ]:
# Cell 4: Define helper function and test questions
def generate_response(model, tokenizer, question, max_tokens=256):
    messages = [{"role": "user", "content": question}]
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            max_new_tokens=max_tokens,
            do_sample=True,
            temperature=0.7,
        )
    response = tokenizer.decode(
        outputs[0][input_ids.shape[-1]:],
        skip_special_tokens=True,
    )
    return response

# Define test questions in global scope
test_questions = [
    "How do I set up an ESP32 MQTT client with TLS encryption?",
    "What is the best protocol for low-power IoT sensor networks?",
    "Explain how to integrate a DHT22 temperature sensor with Home Assistant using ESPHome.",
]

In [ ]:
# Cell 5: Generate and save base model responses
base_responses = []
print("=" * 60)
print("BASE MODEL RESPONSES (before fine-tuning)")
print("=" * 60)
for q in test_questions:
    response = generate_response(model, tokenizer, q)
    base_responses.append(response)
    print(f"\nQ: {q}")
    print(f"A: {response}")
    print("-" * 60)

BASE MODEL RESPONSES (before fine-tuning)


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: How do I set up an ESP32 MQTT client with TLS encryption?
A: Okay, let's break down how to set up an ESP32 MQTT client with TLS encryption. This is a crucial step for secure communication. Here's a comprehensive guide, covering the necessary steps, considerations, and code examples (using the ESP-IDF library):

**1. Prerequisites & Understanding**

* **ESP-IDF:** You'll need the ESP-IDF library.  Install it:
   ```bash
   sudo apt-get update
   sudo apt-get install python3-esp-idf
   ```
* **Python 3:** Ensure Python 3 is installed and accessible.
* **MQTT Broker:** You'll need an MQTT broker running (e.g., Mosquitto, HiveMQ).  Make sure it's accessible from your ESP.
* **TLS Configuration (Important):**  This is the most complex part.  You'll need to configure your ESP-IDF MQTT client to use TLS encryption. There are several ways to do this, each with varying levels of complexity and control.

**2. Recommended Approach: Using the ESP-IDF `mqtt` Library and TLS Configuration**

The

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: What is the best protocol for low-power IoT sensor networks?
A: Okay, let's dive into the best protocols for low-power IoT sensor networks. There's no single "best" – it really depends on the specific sensor type, application requirements (power consumption, range, accuracy), and the complexity you're willing to manage. However, here's a breakdown of the top contenders and their strengths and weaknesses, categorized by their general approach:

**1. Mesh Networking (Most Robust & Flexible)**

* **Protocol:** Zigbee (with Power-Down Modes)
* **How it Works:** Zigbee is a mesh network protocol that uses multiple nodes to extend range and create a self-healing network.  It uses a "power-down" mechanism to conserve energy when nodes are idle.
* **Pros:**
    * **Excellent Range:**  Nodes can communicate with each other, allowing for long-range communication.
    * **Self-Healing:**  If a node fails, the network automatically reroutes traffic through other nodes.
    * **Low Power Consum

In [ ]:
# Cell 6: Load the IoT dataset
from datasets import load_dataset

dataset = load_dataset(
    "electron-rare/mascarade-iot-dataset",
    split="train",
)
print(f"Dataset size: {len(dataset)} samples")
print(f"\nFirst conversation preview:")
for turn in dataset[0]["conversations"]:
    role = turn["from"]
    content = turn["value"][:100]
    print(f"  [{role}]: {content}...")

README.md:   0%|          | 0.00/6.18k [00:00<?, ?B/s]

iot_chat.jsonl: reconstructing file:   0%|          |  0.00B / 11.6MB            

iot_chat.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/6005 [00:00<?, ? examples/s]

Dataset size: 6005 samples

First conversation preview:
  [system]: You are an expert IoT and embedded systems engineer specializing in MQTT protocols, ESP32/ESP8266 wi...
  [human]: Write ESP-IDF code for an MQTT client that publishes sensor data to a broker with TLS and auto-recon...
  [gpt]: ESP-IDF MQTT client with TLS and auto-reconnect:

```c
#include <stdio.h>
#include <string.h>
#inclu...


In [ ]:
# Cell 7: Format dataset for training
def format_conversation(example):
    role_map = {"system": "system", "human": "user", "gpt": "assistant"}
    messages = []
    for turn in example["conversations"]:
        messages.append({
            "role": role_map.get(turn["from"], turn["from"]),
            "content": turn["value"],
        })
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = dataset.map(format_conversation)
print("Formatted sample preview:")
print(dataset[0]["text"][:500])

Map:   0%|          | 0/6005 [00:00<?, ? examples/s]

Formatted sample preview:
<bos><start_of_turn>user
You are an expert IoT and embedded systems engineer specializing in MQTT protocols, ESP32/ESP8266 with ESP-IDF, FreeRTOS, Home Assistant integration, PlatformIO, and smart home automation. You provide production-ready code with proper error handling, secure connections (TLS/mTLS), and energy-efficient designs for battery-powered devices.

Write ESP-IDF code for an MQTT client that publishes sensor data to a broker with TLS and auto-reconnection.<end_of_turn>
<start_of_tu


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

trainer_stats = trainer.train()

Unsloth: Switching to float32 training since model cannot work with float16
Unsloth: transformers renamed `push_to_hub_token` to `hub_token`. Forwarding your value to `hub_token` - update your code when convenient. If you also passed `hub_token` as None, that is its default here and cannot be distinguished from leaving it unset, so `push_to_hub_token` was used; drop `push_to_hub_token` to keep it.
Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/6005 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 6,005 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 13,045,760 of 1,012,931,712 (1.29% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,2.851521
20,1.983878
30,1.829262
40,1.577394
50,1.468346
60,1.519271


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs/checkpoint-60.


In [ ]:
# Cell 10: Compare fine-tuned vs base model
FastLanguageModel.for_inference(model)

print("=" * 60)
print("COMPARISON: BASE vs FINE-TUNED MODEL")
print("=" * 60)
for i, q in enumerate(test_questions):
    finetuned_response = generate_response(model, tokenizer, q, max_tokens=512)
    print(f"\nQ: {q}")
    print(f"\n--- BASE MODEL ---")
    print(base_responses[i])
    print(f"\n--- FINE-TUNED MODEL ---")
    print(finetuned_response)
    print("=" * 60)

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


COMPARISON: BASE vs FINE-TUNED MODEL


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: How do I set up an ESP32 MQTT client with TLS encryption?

--- BASE MODEL ---
Okay, let's break down how to set up an ESP32 MQTT client with TLS encryption. This is a crucial step for secure communication. Here's a comprehensive guide, covering the necessary steps, considerations, and code examples (using the ESP-IDF library):

**1. Prerequisites & Understanding**

* **ESP-IDF:** You'll need the ESP-IDF library.  Install it:
   ```bash
   sudo apt-get update
   sudo apt-get install python3-esp-idf
   ```
* **Python 3:** Ensure Python 3 is installed and accessible.
* **MQTT Broker:** You'll need an MQTT broker running (e.g., Mosquitto, HiveMQ).  Make sure it's accessible from your ESP.
* **TLS Configuration (Important):**  This is the most complex part.  You'll need to configure your ESP-IDF MQTT client to use TLS encryption. There are several ways to do this, each with varying levels of complexity and control.

**2. Recommended Approach: Using the ESP-IDF `mqtt` Library and TLS Con

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: What is the best protocol for low-power IoT sensor networks?

--- BASE MODEL ---
Okay, let's dive into the best protocols for low-power IoT sensor networks. There's no single "best" – it really depends on the specific sensor type, application requirements (power consumption, range, accuracy), and the complexity you're willing to manage. However, here's a breakdown of the top contenders and their strengths and weaknesses, categorized by their general approach:

**1. Mesh Networking (Most Robust & Flexible)**

* **Protocol:** Zigbee (with Power-Down Modes)
* **How it Works:** Zigbee is a mesh network protocol that uses multiple nodes to extend range and create a self-healing network.  It uses a "power-down" mechanism to conserve energy when nodes are idle.
* **Pros:**
    * **Excellent Range:**  Nodes can communicate with each other, allowing for long-range communication.
    * **Self-Healing:**  If a node fails, the network automatically reroutes traffic through other nodes.
    * *

In [ ]:
# Cell 11: Save and push to Hugging Face Hub
model.save_pretrained("iot-expert-gemma3-1b-lora")
tokenizer.save_pretrained("iot-expert-gemma3-1b-lora")

# Replace "your-hf-username" with your actual Hugging Face username
model.push_to_hub("khaoula01/iot-expert-gemma3-1b-lora")
tokenizer.push_to_hub("khaoula01/iot-expert-gemma3-1b-lora")
print("Model pushed to Hugging Face Hub!")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  31%|###       | 16.0MB / 52.2MB            

Validating                    :           |   0%            

No files have been modified since last commit. Skipping to prevent empty commit.


...

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpzapzdsja/tokenizer.json:  72%|#######1  | 23.9MB / 33.4MB            

  ...pzapzdsja/tokenizer.model: 100%|##########| 4.69MB / 4.69MB            

Validating                    :           |   0%            

Model pushed to Hugging Face Hub!


In [ ]:
# Cell 12: Export to GGUF format (add after Cell 11)
# Merge LoRA weights and export to GGUF with Q4_K_M quantization
model.save_pretrained_gguf(
    "iot-expert-gemma3-1b-gguf",
    tokenizer,
    quantization_method="q4_k_m",
)
print("GGUF export complete!")

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.


Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/902 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in iot-expert-gemma3-1b-gguf/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in iot-expert-gemma3-1b-gguf.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

model.safetensors: downloading bytes:           |  0.00B            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:25<00:00, 25.11s/it]


Unsloth: Merge process complete. Saved to `/content/iot-expert-gemma3-1b-gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b11541-mix-2500203 (app-b11541-mix-2500203-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['iot-expert-gemma3-1b-gguf_gguf/gemma-3-1b-it.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...


Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### We removed it in GGUF's chat template for you.


Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['iot-expert-gemma3-1b-gguf_gguf/gemma-3-1b-it.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model iot-expert-gemma3-1b-gguf_gguf/gemma-3-1b-it.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to iot-expert-gemma3-1b-gguf_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f iot-expert-gemma3-1b-gguf_gguf/Modelfile
GGUF export complete!


In [ ]:
# Cell 13: Push GGUF to Hugging Face Hub
# Replace "your-hf-username" with your actual Hugging Face username
model.push_to_hub_gguf(
    "khaoula01/iot-expert-gemma3-1b-gguf",
    tokenizer,
    quantization_method="q4_k_m",
)
print("GGUF model pushed to Hugging Face Hub!")

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.


Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...


Unsloth: Restored added_tokens_decoder metadata in /tmp/unsloth_gguf_shtsz1ij/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /tmp/unsloth_gguf_shtsz1ij.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...gemma-3-1b-it.Q4_K_M.gguf:   2%|1         | 15.7MB /  806MB            

Validating                    :           |   0%            

Unsloth: Successfully uploaded GGUF to https://huggingface.co/khaoula01/iot-expert-gemma3-1b-gguf
Unsloth: Cleaning up temporary files...
GGUF model pushed to Hugging Face Hub!
